ROUND 1 - 100 imnage having >85% confidence extraction

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Install required libraries
!pip install -q torch torchvision timm opencv-python tqdm

# Define your base Drive directory (Update this if your folder name is different)
BASE_DRIVE_DIR = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE"

# Verify files exist
print("\n--- Verifying Assets ---")
print("Weights found:", os.path.exists(os.path.join(BASE_DRIVE_DIR, "best.pt.zip")))
print("Augmented Dataset found:", os.path.exists(os.path.join(BASE_DRIVE_DIR, "augmented_gt")))


Mounted at /content/drive

--- Verifying Assets ---
Weights found: True
Augmented Dataset found: True


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip=None):
        x = self.upsample(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = nn.functional.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class TransUNetHybrid(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=None)

        self.encoder1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.encoder2 = nn.Sequential(resnet.maxpool, resnet.layer1)
        self.encoder3 = resnet.layer2
        self.encoder4 = resnet.layer3
        self.encoder5 = resnet.layer4

        self.transformer_bridge = nn.Sequential(
            nn.Conv2d(2048, 512, kernel_size=1),
            nn.GroupNorm(32, 512),
            nn.ReLU(inplace=True)
        )

        self.dec1 = DecoderBlock(512, 1024, 256)
        self.dec2 = DecoderBlock(256, 512, 128)
        self.dec3 = DecoderBlock(128, 256, 64)
        self.dec4 = DecoderBlock(64, 0, 32)

        self.final_head = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        c1 = self.encoder1(x)
        c2 = self.encoder2(c1)
        c3 = self.encoder3(c2)
        c4 = self.encoder4(c3)
        c5 = self.encoder5(c4)

        bottleneck = self.transformer_bridge(c5)

        x = self.dec1(bottleneck, c4)
        x = self.dec2(x, c3)
        x = self.dec3(x, c2)
        x = self.dec4(x)

        out = self.final_head(x)
        out = nn.functional.interpolate(out, size=(512, 512), mode='bilinear', align_corners=True)

        return out

In [ ]:
import cv2
import numpy as np
import shutil
from tqdm.notebook import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WEIGHTS_PATH = os.path.join(BASE_DRIVE_DIR, "best.pt.zip")
UNLABELED_DIR = os.path.join(BASE_DRIVE_DIR, "Glofeagles Dataset unnanotated")
PSEUDO_DIR = os.path.join(BASE_DRIVE_DIR, "pseudo_labeled_round_1")

os.makedirs(os.path.join(PSEUDO_DIR, "images"), exist_ok=True)
os.makedirs(os.path.join(PSEUDO_DIR, "labels"), exist_ok=True)

print("Loading model for inference...")
model = TransUNetHybrid().to(DEVICE)
model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
model.eval()

unlabeled_files = [f for f in os.listdir(UNLABELED_DIR) if f.endswith(('.png', '.jpg'))]
image_scores = []

print(f"Scoring {len(unlabeled_files)} unlabeled images...")
with torch.no_grad():
    for filename in tqdm(unlabeled_files):
        img_path = os.path.join(UNLABELED_DIR, filename)
        img = cv2.imread(img_path)
        if img is None: continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor_img = torch.tensor(img_rgb.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

        probs = torch.sigmoid(model(tensor_img)).squeeze()
        binary_mask = (probs > 0.5).float()
        lake_pixels = probs[binary_mask == 1.0]

        # Calculate confidence (must have > 50 pixels to be considered a real lake)
        confidence = lake_pixels.mean().item() * 100 if len(lake_pixels) > 50 else 0.0

        image_scores.append({
            "filename": filename,
            "confidence": confidence,
            "mask": binary_mask.cpu().numpy()
        })

# Sort and filter for > 85% confidence
image_scores.sort(key=lambda x: x["confidence"], reverse=True)
confident_images = [x for x in image_scores if x["confidence"] >= 85.0]
top_k = min(100, len(confident_images))

print(f"\nExtracting top {top_k} highly confident images to pseudo-label folder...")
for i in tqdm(range(top_k)):
    data = confident_images[i]
    filename = data["filename"]

    # Save predicted mask as the new label
    cv2.imwrite(os.path.join(PSEUDO_DIR, "labels", filename), (data["mask"] * 255).astype(np.uint8))

    # Move original image
    shutil.copy(os.path.join(UNLABELED_DIR, filename), os.path.join(PSEUDO_DIR, "images", filename))

    # Remove from unlabeled pool
    os.remove(os.path.join(UNLABELED_DIR, filename))

print(f"Success! {top_k} new training samples generated.")
print(f"Unlabeled images remaining: {len(os.listdir(UNLABELED_DIR))}")

Loading model for inference...
Scoring 516 unlabeled images...


  0%|          | 0/516 [00:00<?, ?it/s]


Extracting top 100 highly confident images to pseudo-label folder...


  0%|          | 0/100 [00:00<?, ?it/s]

Success! 100 new training samples generated.
Unlabeled images remaining: 416


ROUND 1 - DATASET + 100*10 images add and Training

In [ ]:
import os
import cv2
import numpy as np
import albumentations as A
import zipfile
from tqdm.notebook import tqdm

# --- 1. EXACT PATHS ---
BASE_DRIVE_DIR = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE"
WEIGHTS_ZIP = os.path.join(BASE_DRIVE_DIR, "best.pt.zip")
PSEUDO_DIR = os.path.join(BASE_DRIVE_DIR, "pseudo_labeled_round_1")
MASTER_DATASET_DIR = os.path.join(BASE_DRIVE_DIR, "augmented_gt")
LOCAL_WEIGHTS_PATH = "/content/best.pt" # We extract to Colab's local drive for safe loading

# --- 2. EXTRACT WEIGHTS ---
print("Extracting best.pt.zip...")
if os.path.exists(WEIGHTS_ZIP):
    with zipfile.ZipFile(WEIGHTS_ZIP, 'r') as zip_ref:
        zip_ref.extractall("/content/")
    print("Weights extracted safely.")
else:
    print(f"Error: Could not find {WEIGHTS_ZIP}")

# --- 3. AUGMENTATION PIPELINE (Updated for Albumentations v1.4+) ---
transform = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.8),
    A.HueSaturationValue(hue_shift_limit=30, sat_shift_limit=40, val_shift_limit=20, p=0.8),
    A.OneOf([
        A.Affine(rotate=0, p=1.0), A.Affine(rotate=45, p=1.0), A.Affine(rotate=90, p=1.0),
        A.Affine(rotate=135, p=1.0), A.Affine(rotate=180, p=1.0), A.Affine(rotate=225, p=1.0),
        A.Affine(rotate=270, p=1.0), A.Affine(rotate=315, p=1.0),
    ], p=0.8),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

    # Modernized parameters
    A.RandomFog(p=0.3),
    A.GaussNoise(p=0.3),
    A.ElasticTransform(alpha=1, sigma=50, p=0.3), # Removed deprecated alpha_affine
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 32), hole_width_range=(8, 32), fill=0, mask_fill=0, p=0.1),
    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.1),
    A.RandomResizedCrop(size=(512, 512), scale=(0.85, 1.0), ratio=(0.9, 1.1), p=0.1) # Combined height/width into size tuple
])

# --- 4. MERGE DIRECTLY INTO MASTER DATASET ---
pseudo_images_dir = os.path.join(PSEUDO_DIR, "images")
pseudo_labels_dir = os.path.join(PSEUDO_DIR, "labels")

master_images_dir = os.path.join(MASTER_DATASET_DIR, "images")
master_labels_dir = os.path.join(MASTER_DATASET_DIR, "labels")

pseudo_files = [f for f in os.listdir(pseudo_images_dir) if f.endswith(('.png', '.jpg'))]

print(f"Augmenting {len(pseudo_files)} pseudo-labeled images into the master dataset...")
for filename in tqdm(pseudo_files):
    img_path = os.path.join(pseudo_images_dir, filename)
    lbl_path = os.path.join(pseudo_labels_dir, filename)

    image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)

    # Save original pseudo-label with a prefix to prevent name clashes
    base_name = os.path.splitext(filename)[0]
    cv2.imwrite(os.path.join(master_images_dir, f"pseudo_{base_name}_aug_0.png"), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(master_labels_dir, f"pseudo_{base_name}_aug_0.png"), mask)

    # Generate 10 variations
    for i in range(1, 11):
        augmented = transform(image=image, mask=mask)
        new_filename = f"pseudo_{base_name}_aug_{i}.png"

        cv2.imwrite(os.path.join(master_images_dir, new_filename), cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR))
        cv2.imwrite(os.path.join(master_labels_dir, new_filename), augmented['mask'])

print(f"Success! Master dataset now contains {len(os.listdir(master_images_dir))} total images.")

Extracting best.pt.zip...
Weights extracted safely.
Augmenting 100 pseudo-labeled images into the master dataset...


/tmp/ipykernel_582/1707628964.py:40: UserWarning: Argument(s) 'mask_fill' are not valid for transform CoarseDropout
  A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 32), hole_width_range=(8, 32), fill=0, mask_fill=0, p=0.1),


  0%|          | 0/100 [00:00<?, ?it/s]

Success! Master dataset now contains 1760 total images.


In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.models as models
from tqdm.notebook import tqdm
import shutil

# --- 1. FINE-TUNING HYPERPARAMETERS & PATHS ---
BASE_DRIVE_DIR = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE"
MASTER_DATASET_DIR = os.path.join(BASE_DRIVE_DIR, "augmented_gt")
NEW_WEIGHTS_DIR = os.path.join(BASE_DRIVE_DIR, "runs/train/fine_tuned/weights")

os.makedirs(NEW_WEIGHTS_DIR, exist_ok=True)

EPOCHS = 12                  # Fine-tuning requires fewer epochs
BATCH_SIZE = 6
LEARNING_RATE = 1e-5         # Low learning rate to prevent Catastrophic Forgetting
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. DATASET DEFINITION ---
class GlacialLakeDataset(Dataset):
    def __init__(self, base_dir):
        self.images_dir = os.path.join(base_dir, "images")
        self.labels_dir = os.path.join(base_dir, "labels")
        self.filenames = [f for f in os.listdir(self.images_dir) if f.endswith(('.png', '.jpg'))]
        self.filenames.sort()

    def __len__(self): return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        img_path = os.path.join(self.images_dir, filename)
        lbl_path = os.path.join(self.labels_dir, filename)

        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        image = torch.tensor(image.astype(np.float32) / 255.0).permute(2, 0, 1)

        mask = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)
        mask = torch.tensor((mask > 127).astype(np.float32)).unsqueeze(0)

        return image, mask

# --- 3. ARCHITECTURE DEFINITION ---
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )
    def forward(self, x, skip=None):
        x = self.upsample(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = nn.functional.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class TransUNetHybrid(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=None)
        self.encoder1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.encoder2 = nn.Sequential(resnet.maxpool, resnet.layer1)
        self.encoder3 = resnet.layer2
        self.encoder4 = resnet.layer3
        self.encoder5 = resnet.layer4
        self.transformer_bridge = nn.Sequential(nn.Conv2d(2048, 512, kernel_size=1), nn.GroupNorm(32, 512), nn.ReLU(inplace=True))
        self.dec1, self.dec2 = DecoderBlock(512, 1024, 256), DecoderBlock(256, 512, 128)
        self.dec3, self.dec4 = DecoderBlock(128, 256, 64), DecoderBlock(64, 0, 32)
        self.final_head = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        c1, c2, c3, c4, c5 = self.encoder1(x), self.encoder2(self.encoder1(x)), self.encoder3(self.encoder2(self.encoder1(x))), self.encoder4(self.encoder3(self.encoder2(self.encoder1(x)))), self.encoder5(self.encoder4(self.encoder3(self.encoder2(self.encoder1(x)))))
        x = self.dec4(self.dec3(self.dec2(self.dec1(self.transformer_bridge(c5), c4), c3), c2))
        return nn.functional.interpolate(self.final_head(x), size=(512, 512), mode='bilinear', align_corners=True)

# --- 4. LOSS FUNCTIONS ---
class HybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, pred, target):
        p = torch.sigmoid(pred)
        dice = 1.0 - (2. * (p * target).sum(dim=(2, 3)) + 1e-5) / (p.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + 1e-5)
        return self.bce(pred, target) + dice.mean()

# --- 5. INITIALIZE AND LOAD EXISTING KNOWLEDGE ---
print("Preparing Datasets...")
dataset = GlacialLakeDataset(MASTER_DATASET_DIR)
train_set, val_set = random_split(dataset, [int(0.8 * len(dataset)), len(dataset) - int(0.8 * len(dataset))], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

print("Loading Brain...")
model = TransUNetHybrid().to(DEVICE)

# Fetch the perfectly formatted zip from Drive, copy locally, and rename to .pt
drive_weights_path = os.path.join(BASE_DRIVE_DIR, "best.pt.zip")
local_weights_path = "/content/structured_weights.pt"

if os.path.exists(drive_weights_path):
    print("Fetching structured weights from Drive...")
    shutil.copy(drive_weights_path, local_weights_path)

    try:
        model.load_state_dict(torch.load(local_weights_path, map_location=DEVICE))
        print("🚀 Previous knowledge successfully injected!")
    except Exception as e:
        print(f"❌ Failed to load weights. Error: {e}")
        print("Please ensure your 'archive' folder is at the absolute root of the zip file.")
else:
    print(f"WARNING: Could not find {drive_weights_path}. Training will start from scratch!")

criterion = HybridLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# --- 6. FINE-TUNING EXECUTION ---
best_loss = float('inf')
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_train_loss = 0.0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch} Train", leave=False):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), masks)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()

    scheduler.step()

    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch} Val", leave=False):
            running_val_loss += criterion(model(images.to(DEVICE)), masks.to(DEVICE)).item()

    val_loss = running_val_loss / len(val_loader)
    print(f"Epoch {epoch}/{EPOCHS} -> Train Loss: {running_train_loss / len(train_loader):.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_loss:
        print(f"✨ Validation improved! Saving new weights to {NEW_WEIGHTS_DIR}...")
        best_loss = val_loss
        torch.save(model.state_dict(), os.path.join(NEW_WEIGHTS_DIR, "best_finetuned.pt"))

Preparing Datasets...
Loading Brain...
Fetching structured weights from Drive...
🚀 Previous knowledge successfully injected!


Epoch 1 Train:   0%|          | 0/234 [00:00<?, ?it/s]

Epoch 1 Val:   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 1/12 -> Train Loss: 0.4175 | Val Loss: 0.4741
✨ Validation improved! Saving new weights to /content/drive/MyDrive/GLOFEAGLES CHALLENGE/runs/train/fine_tuned/weights...


Epoch 2 Train:   0%|          | 0/234 [00:00<?, ?it/s]

...

Epoch 10/12 -> Train Loss: 0.2597 | Val Loss: 0.3814
✨ Validation improved! Saving new weights to /content/drive/MyDrive/GLOFEAGLES CHALLENGE/runs/train/fine_tuned/weights...


Epoch 11 Train:   0%|          | 0/234 [00:00<?, ?it/s]

Epoch 11 Val:   0%|          | 0/59 [00:00<?, ?it/s]

Epoch 11/12 -> Train Loss: 0.2637 | Val Loss: 0.3813
✨ Validation improved! Saving new weights to /content/drive/MyDrive/GLOFEAGLES CHALLENGE/runs/train/fine_tuned/weights...


Epoch 12 Train:   0%|          | 0/234 [00:00<?, ?it/s]

ROUND 2 - extracting top-100 >88% confidence images

In [ ]:
import os
import cv2
import torch
import numpy as np
import shutil
from tqdm.notebook import tqdm

# --- 1. ROUND 2 PATHS ---
BASE_DRIVE_DIR = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE"
# Pointing to the new, smarter brain!
WEIGHTS_PATH = os.path.join(BASE_DRIVE_DIR, "runs/train/fine_tuned/weights/best_finetuned.pt")
UNLABELED_DIR = os.path.join(BASE_DRIVE_DIR, "Glofeagles Dataset unnanotated")
# Creating a new folder for Round 2
PSEUDO_DIR = os.path.join(BASE_DRIVE_DIR, "pseudo_labeled_round_2")

os.makedirs(os.path.join(PSEUDO_DIR, "images"), exist_ok=True)
os.makedirs(os.path.join(PSEUDO_DIR, "labels"), exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. LOAD FINE-TUNED MODEL ---
print("Loading fine-tuned model for Round 2 inference...")
model = TransUNetHybrid().to(DEVICE)
if os.path.exists(WEIGHTS_PATH):
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
    model.eval()
    print("✅ Model loaded successfully.")
else:
    raise FileNotFoundError(f"Could not find fine-tuned weights at {WEIGHTS_PATH}")

# --- 3. INFERENCE ---
unlabeled_files = [f for f in os.listdir(UNLABELED_DIR) if f.endswith(('.png', '.jpg'))]
image_scores = []

print(f"Scoring the remaining {len(unlabeled_files)} unlabeled images...")
with torch.no_grad():
    for filename in tqdm(unlabeled_files):
        img_path = os.path.join(UNLABELED_DIR, filename)
        img = cv2.imread(img_path)
        if img is None: continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor_img = torch.tensor(img_rgb.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

        probs = torch.sigmoid(model(tensor_img)).squeeze()
        binary_mask = (probs > 0.5).float()
        lake_pixels = probs[binary_mask == 1.0]

        confidence = lake_pixels.mean().item() * 100 if len(lake_pixels) > 50 else 0.0

        image_scores.append({
            "filename": filename,
            "confidence": confidence,
            "mask": binary_mask.cpu().numpy()
        })

# --- 4. STRICT 88% FILTER ---
image_scores.sort(key=lambda x: x["confidence"], reverse=True)
confident_images = [x for x in image_scores if x["confidence"] >= 88.0]

# Take up to 100 images that pass the new strict threshold
top_k = min(100, len(confident_images))

print(f"\nFound {len(confident_images)} images with >= 88% confidence. Extracting top {top_k}...")
for i in tqdm(range(top_k)):
    data = confident_images[i]
    filename = data["filename"]

    cv2.imwrite(os.path.join(PSEUDO_DIR, "labels", filename), (data["mask"] * 255).astype(np.uint8))
    shutil.copy(os.path.join(UNLABELED_DIR, filename), os.path.join(PSEUDO_DIR, "images", filename))
    os.remove(os.path.join(UNLABELED_DIR, filename))

print(f"Success! Round 2 generated {top_k} new high-quality samples.")
print(f"Remaining unlabeled pool: {len(os.listdir(UNLABELED_DIR))}")

Loading fine-tuned model for Round 2 inference...
✅ Model loaded successfully.
Scoring the remaining 416 unlabeled images...


  0%|          | 0/416 [00:00<?, ?it/s]


Found 291 images with >= 88% confidence. Extracting top 100...


  0%|          | 0/100 [00:00<?, ?it/s]

Success! Round 2 generated 100 new high-quality samples.
Remaining unlabeled pool: 316


ROUND 2 :1760 + 100*10 + 100 = 2860 images addition and Training

In [ ]:
import os
import cv2
import numpy as np
import albumentations as A
import zipfile
from tqdm.notebook import tqdm

# --- 1. EXACT PATHS ---
BASE_DRIVE_DIR = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE"
WEIGHTS_ZIP = os.path.join(BASE_DRIVE_DIR, "best.pt.zip")
PSEUDO_DIR = os.path.join(BASE_DRIVE_DIR, "pseudo_labeled_round_2")
MASTER_DATASET_DIR = os.path.join(BASE_DRIVE_DIR, "augmented_gt")
LOCAL_WEIGHTS_PATH = "/content/best.pt" # We extract to Colab's local drive for safe loading

# --- 2. EXTRACT WEIGHTS ---
print("Extracting best.pt.zip...")
if os.path.exists(WEIGHTS_ZIP):
    with zipfile.ZipFile(WEIGHTS_ZIP, 'r') as zip_ref:
        zip_ref.extractall("/content/")
    print("Weights extracted safely.")
else:
    print(f"Error: Could not find {WEIGHTS_ZIP}")

# --- 3. AUGMENTATION PIPELINE (Updated for Albumentations v1.4+) ---
transform = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.8),
    A.HueSaturationValue(hue_shift_limit=30, sat_shift_limit=40, val_shift_limit=20, p=0.8),
    A.OneOf([
        A.Affine(rotate=0, p=1.0), A.Affine(rotate=45, p=1.0), A.Affine(rotate=90, p=1.0),
        A.Affine(rotate=135, p=1.0), A.Affine(rotate=180, p=1.0), A.Affine(rotate=225, p=1.0),
        A.Affine(rotate=270, p=1.0), A.Affine(rotate=315, p=1.0),
    ], p=0.8),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

    # Modernized parameters
    A.RandomFog(p=0.3),
    A.GaussNoise(p=0.3),
    A.ElasticTransform(alpha=1, sigma=50, p=0.3), # Removed deprecated alpha_affine
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 32), hole_width_range=(8, 32), fill=0, mask_fill=0, p=0.1),
    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.1),
    A.RandomResizedCrop(size=(512, 512), scale=(0.85, 1.0), ratio=(0.9, 1.1), p=0.1) # Combined height/width into size tuple
])

# --- 4. MERGE DIRECTLY INTO MASTER DATASET ---
pseudo_images_dir = os.path.join(PSEUDO_DIR, "images")
pseudo_labels_dir = os.path.join(PSEUDO_DIR, "labels")

master_images_dir = os.path.join(MASTER_DATASET_DIR, "images")
master_labels_dir = os.path.join(MASTER_DATASET_DIR, "labels")

pseudo_files = [f for f in os.listdir(pseudo_images_dir) if f.endswith(('.png', '.jpg'))]

print(f"Augmenting {len(pseudo_files)} pseudo-labeled images into the master dataset...")
for filename in tqdm(pseudo_files):
    img_path = os.path.join(pseudo_images_dir, filename)
    lbl_path = os.path.join(pseudo_labels_dir, filename)

    image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)

    # Save original pseudo-label with a prefix to prevent name clashes
    base_name = os.path.splitext(filename)[0]
    cv2.imwrite(os.path.join(master_images_dir, f"pseudo_{base_name}_aug_0.png"), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(master_labels_dir, f"pseudo_{base_name}_aug_0.png"), mask)

    # Generate 10 variations
    for i in range(1, 11):
        augmented = transform(image=image, mask=mask)
        new_filename = f"pseudo_{base_name}_aug_{i}.png"

        cv2.imwrite(os.path.join(master_images_dir, new_filename), cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR))
        cv2.imwrite(os.path.join(master_labels_dir, new_filename), augmented['mask'])

print(f"Success! Master dataset now contains {len(os.listdir(master_images_dir))} total images.")

Extracting best.pt.zip...
Weights extracted safely.
Augmenting 100 pseudo-labeled images into the master dataset...


/tmp/ipykernel_1878/2647973875.py:40: UserWarning: Argument(s) 'mask_fill' are not valid for transform CoarseDropout
  A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 32), hole_width_range=(8, 32), fill=0, mask_fill=0, p=0.1),


  0%|          | 0/100 [00:00<?, ?it/s]

Success! Master dataset now contains 2860 total images.


In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.models as models
from tqdm.notebook import tqdm
import shutil

# --- 1. FINE-TUNING HYPERPARAMETERS & PATHS ---
BASE_DRIVE_DIR = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE"
MASTER_DATASET_DIR = os.path.join(BASE_DRIVE_DIR, "augmented_gt")
NEW_WEIGHTS_DIR = os.path.join(BASE_DRIVE_DIR, "runs/train/fine_tuned/weights")

os.makedirs(NEW_WEIGHTS_DIR, exist_ok=True)

EPOCHS = 8                  # Fine-tuning requires fewer epochs
BATCH_SIZE = 4
LEARNING_RATE = 1e-5         # Low learning rate to prevent Catastrophic Forgetting
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. DATASET DEFINITION ---
class GlacialLakeDataset(Dataset):
    def __init__(self, base_dir):
        self.images_dir = os.path.join(base_dir, "images")
        self.labels_dir = os.path.join(base_dir, "labels")
        self.filenames = [f for f in os.listdir(self.images_dir) if f.endswith(('.png', '.jpg'))]
        self.filenames.sort()

    def __len__(self): return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        img_path = os.path.join(self.images_dir, filename)
        lbl_path = os.path.join(self.labels_dir, filename)

        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        image = torch.tensor(image.astype(np.float32) / 255.0).permute(2, 0, 1)

        mask = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)
        mask = torch.tensor((mask > 127).astype(np.float32)).unsqueeze(0)

        return image, mask

# --- 3. ARCHITECTURE DEFINITION ---
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )
    def forward(self, x, skip=None):
        x = self.upsample(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = nn.functional.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class TransUNetHybrid(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=None)
        self.encoder1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.encoder2 = nn.Sequential(resnet.maxpool, resnet.layer1)
        self.encoder3 = resnet.layer2
        self.encoder4 = resnet.layer3
        self.encoder5 = resnet.layer4
        self.transformer_bridge = nn.Sequential(nn.Conv2d(2048, 512, kernel_size=1), nn.GroupNorm(32, 512), nn.ReLU(inplace=True))
        self.dec1, self.dec2 = DecoderBlock(512, 1024, 256), DecoderBlock(256, 512, 128)
        self.dec3, self.dec4 = DecoderBlock(128, 256, 64), DecoderBlock(64, 0, 32)
        self.final_head = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        c1, c2, c3, c4, c5 = self.encoder1(x), self.encoder2(self.encoder1(x)), self.encoder3(self.encoder2(self.encoder1(x))), self.encoder4(self.encoder3(self.encoder2(self.encoder1(x)))), self.encoder5(self.encoder4(self.encoder3(self.encoder2(self.encoder1(x)))))
        x = self.dec4(self.dec3(self.dec2(self.dec1(self.transformer_bridge(c5), c4), c3), c2))
        return nn.functional.interpolate(self.final_head(x), size=(512, 512), mode='bilinear', align_corners=True)

# --- 4. LOSS FUNCTIONS ---
class HybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, pred, target):
        p = torch.sigmoid(pred)
        dice = 1.0 - (2. * (p * target).sum(dim=(2, 3)) + 1e-5) / (p.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + 1e-5)
        return self.bce(pred, target) + dice.mean()

# --- 5. INITIALIZE AND LOAD EXISTING KNOWLEDGE ---
print("Preparing Datasets...")
dataset = GlacialLakeDataset(MASTER_DATASET_DIR)
train_set, val_set = random_split(dataset, [int(0.8 * len(dataset)), len(dataset) - int(0.8 * len(dataset))], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

print("Loading Brain...")
model = TransUNetHybrid().to(DEVICE)

# Fetch the perfectly formatted zip from Drive, copy locally, and rename to .pt
drive_weights_path = os.path.join(BASE_DRIVE_DIR, "best.pt.zip")
local_weights_path = "/content/structured_weights.pt"

if os.path.exists(drive_weights_path):
    print("Fetching structured weights from Drive...")
    shutil.copy(drive_weights_path, local_weights_path)

    try:
        model.load_state_dict(torch.load(local_weights_path, map_location=DEVICE))
        print("🚀 Previous knowledge successfully injected!")
    except Exception as e:
        print(f"❌ Failed to load weights. Error: {e}")
        print("Please ensure your 'archive' folder is at the absolute root of the zip file.")
else:
    print(f"WARNING: Could not find {drive_weights_path}. Training will start from scratch!")

criterion = HybridLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# --- 6. FINE-TUNING EXECUTION ---
best_loss = float('inf')
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_train_loss = 0.0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch} Train", leave=False):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), masks)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()

    scheduler.step()

    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch} Val", leave=False):
            running_val_loss += criterion(model(images.to(DEVICE)), masks.to(DEVICE)).item()

    val_loss = running_val_loss / len(val_loader)
    print(f"Epoch {epoch}/{EPOCHS} -> Train Loss: {running_train_loss / len(train_loader):.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_loss:
        print(f"✨ Validation improved! Saving new weights to {NEW_WEIGHTS_DIR}...")
        best_loss = val_loss
        torch.save(model.state_dict(), os.path.join(NEW_WEIGHTS_DIR, "best_finetuned.pt"))

Preparing Datasets...
Loading Brain...
Fetching structured weights from Drive...
🚀 Previous knowledge successfully injected!


Epoch 1 Train:   0%|          | 0/572 [00:00<?, ?it/s]

Epoch 1 Val:   0%|          | 0/143 [00:00<?, ?it/s]

Epoch 1/8 -> Train Loss: 0.4937 | Val Loss: 0.4750
✨ Validation improved! Saving new weights to /content/drive/MyDrive/GLOFEAGLES CHALLENGE/runs/train/fine_tuned/weights...


Epoch 2 Train:   0%|          | 0/572 [00:00<?, ?it/s]

...

Epoch 7 Val:   0%|          | 0/143 [00:00<?, ?it/s]

Epoch 7/8 -> Train Loss: 0.3366 | Val Loss: 0.4261


Epoch 8 Train:   0%|          | 0/572 [00:00<?, ?it/s]

Epoch 8 Val:   0%|          | 0/143 [00:00<?, ?it/s]

Epoch 8/8 -> Train Loss: 0.3252 | Val Loss: 0.4003


ROUND 3 : Extraction of images >90%

In [ ]:
import os
import cv2
import torch
import numpy as np
import shutil
from tqdm.notebook import tqdm

# --- 1. ROUND 3 PATHS ---
BASE_DRIVE_DIR = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE"
WEIGHTS_PATH = os.path.join(BASE_DRIVE_DIR, "runs/train/fine_tuned/weights/best_finetuned.pt")
UNLABELED_DIR = os.path.join(BASE_DRIVE_DIR, "Glofeagles Dataset unnanotated")
# Creating a new folder for Round 3
PSEUDO_DIR = os.path.join(BASE_DRIVE_DIR, "pseudo_labeled_round_3")

os.makedirs(os.path.join(PSEUDO_DIR, "images"), exist_ok=True)
os.makedirs(os.path.join(PSEUDO_DIR, "labels"), exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. LOAD LATEST FINE-TUNED MODEL ---
print("Loading fine-tuned model for Round 3 inference...")
model = TransUNetHybrid().to(DEVICE)
if os.path.exists(WEIGHTS_PATH):
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
    model.eval()
    print("✅ Model loaded successfully.")
else:
    raise FileNotFoundError(f"Could not find fine-tuned weights at {WEIGHTS_PATH}")

# --- 3. INFERENCE ---
unlabeled_files = [f for f in os.listdir(UNLABELED_DIR) if f.endswith(('.png', '.jpg'))]
image_scores = []

print(f"Scoring the remaining {len(unlabeled_files)} unlabeled images...")
with torch.no_grad():
    for filename in tqdm(unlabeled_files):
        img_path = os.path.join(UNLABELED_DIR, filename)
        img = cv2.imread(img_path)
        if img is None: continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor_img = torch.tensor(img_rgb.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

        probs = torch.sigmoid(model(tensor_img)).squeeze()
        binary_mask = (probs > 0.5).float()
        lake_pixels = probs[binary_mask == 1.0]

        confidence = lake_pixels.mean().item() * 100 if len(lake_pixels) > 50 else 0.0

        image_scores.append({
            "filename": filename,
            "confidence": confidence,
            "mask": binary_mask.cpu().numpy()
        })

# --- 4. STRICT 90% FILTER (ALL QUALIFYING IMAGES) ---
image_scores.sort(key=lambda x: x["confidence"], reverse=True)
confident_images = [x for x in image_scores if x["confidence"] >= 90.0]

print(f"\nFound {len(confident_images)} images with >= 90% confidence. Extracting ALL of them...")
for data in tqdm(confident_images, desc="Extracting Round 3 files"):
    filename = data["filename"]

    cv2.imwrite(os.path.join(PSEUDO_DIR, "labels", filename), (data["mask"] * 255).astype(np.uint8))
    shutil.copy(os.path.join(UNLABELED_DIR, filename), os.path.join(PSEUDO_DIR, "images", filename))
    os.remove(os.path.join(UNLABELED_DIR, filename))

print(f"Success! Round 3 generated {len(confident_images)} new high-quality samples.")
print(f"Remaining unlabeled pool: {len(os.listdir(UNLABELED_DIR))}")

Loading fine-tuned model for Round 3 inference...
✅ Model loaded successfully.
Scoring the remaining 316 unlabeled images...


  0%|          | 0/316 [00:00<?, ?it/s]


Found 177 images with >= 90% confidence. Extracting ALL of them...


Extracting Round 3 files:   0%|          | 0/177 [00:00<?, ?it/s]

Success! Round 3 generated 177 new high-quality samples.
Remaining unlabeled pool: 139


ROUND 3: 2860 + 177*10 + 177 = 4807 images and Training

In [ ]:
import os
import cv2
import numpy as np
import albumentations as A
import zipfile
from tqdm.notebook import tqdm

# --- 1. EXACT PATHS ---
BASE_DRIVE_DIR = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE"
WEIGHTS_ZIP = os.path.join(BASE_DRIVE_DIR, "best.pt.zip")
PSEUDO_DIR = os.path.join(BASE_DRIVE_DIR, "pseudo_labeled_round_3")
MASTER_DATASET_DIR = os.path.join(BASE_DRIVE_DIR, "augmented_gt")
LOCAL_WEIGHTS_PATH = "/content/best.pt" # We extract to Colab's local drive for safe loading

# --- 2. EXTRACT WEIGHTS ---
print("Extracting best.pt.zip...")
if os.path.exists(WEIGHTS_ZIP):
    with zipfile.ZipFile(WEIGHTS_ZIP, 'r') as zip_ref:
        zip_ref.extractall("/content/")
    print("Weights extracted safely.")
else:
    print(f"Error: Could not find {WEIGHTS_ZIP}")

# --- 3. AUGMENTATION PIPELINE (Updated for Albumentations v1.4+) ---
transform = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.8),
    A.HueSaturationValue(hue_shift_limit=30, sat_shift_limit=40, val_shift_limit=20, p=0.8),
    A.OneOf([
        A.Affine(rotate=0, p=1.0), A.Affine(rotate=45, p=1.0), A.Affine(rotate=90, p=1.0),
        A.Affine(rotate=135, p=1.0), A.Affine(rotate=180, p=1.0), A.Affine(rotate=225, p=1.0),
        A.Affine(rotate=270, p=1.0), A.Affine(rotate=315, p=1.0),
    ], p=0.8),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

    # Modernized parameters
    A.RandomFog(p=0.3),
    A.GaussNoise(p=0.3),
    A.ElasticTransform(alpha=1, sigma=50, p=0.3), # Removed deprecated alpha_affine
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 32), hole_width_range=(8, 32), fill=0, mask_fill=0, p=0.1),
    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.1),
    A.RandomResizedCrop(size=(512, 512), scale=(0.85, 1.0), ratio=(0.9, 1.1), p=0.1) # Combined height/width into size tuple
])

# --- 4. MERGE DIRECTLY INTO MASTER DATASET ---
pseudo_images_dir = os.path.join(PSEUDO_DIR, "images")
pseudo_labels_dir = os.path.join(PSEUDO_DIR, "labels")

master_images_dir = os.path.join(MASTER_DATASET_DIR, "images")
master_labels_dir = os.path.join(MASTER_DATASET_DIR, "labels")

pseudo_files = [f for f in os.listdir(pseudo_images_dir) if f.endswith(('.png', '.jpg'))]

print(f"Augmenting {len(pseudo_files)} pseudo-labeled images into the master dataset...")
for filename in tqdm(pseudo_files):
    img_path = os.path.join(pseudo_images_dir, filename)
    lbl_path = os.path.join(pseudo_labels_dir, filename)

    image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)

    # Save original pseudo-label with a prefix to prevent name clashes
    base_name = os.path.splitext(filename)[0]
    cv2.imwrite(os.path.join(master_images_dir, f"pseudo_{base_name}_aug_0.png"), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(master_labels_dir, f"pseudo_{base_name}_aug_0.png"), mask)

    # Generate 10 variations
    for i in range(1, 11):
        augmented = transform(image=image, mask=mask)
        new_filename = f"pseudo_{base_name}_aug_{i}.png"

        cv2.imwrite(os.path.join(master_images_dir, new_filename), cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR))
        cv2.imwrite(os.path.join(master_labels_dir, new_filename), augmented['mask'])

print(f"Success! Master dataset now contains {len(os.listdir(master_images_dir))} total images.")

Extracting best.pt.zip...
Weights extracted safely.
Augmenting 177 pseudo-labeled images into the master dataset...


/tmp/ipykernel_57039/2277449677.py:40: UserWarning: Argument(s) 'mask_fill' are not valid for transform CoarseDropout
  A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 32), hole_width_range=(8, 32), fill=0, mask_fill=0, p=0.1),


  0%|          | 0/177 [00:00<?, ?it/s]

Success! Master dataset now contains 4807 total images.


In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.models as models
from tqdm.notebook import tqdm
import shutil

# --- 1. FINE-TUNING HYPERPARAMETERS & PATHS ---
BASE_DRIVE_DIR = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE"
MASTER_DATASET_DIR = os.path.join(BASE_DRIVE_DIR, "augmented_gt")
NEW_WEIGHTS_DIR = os.path.join(BASE_DRIVE_DIR, "runs/train/fine_tuned/weights")

os.makedirs(NEW_WEIGHTS_DIR, exist_ok=True)

EPOCHS = 5                  # Fine-tuning requires fewer epochs
BATCH_SIZE = 4
LEARNING_RATE = 1e-5         # Low learning rate to prevent Catastrophic Forgetting
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. DATASET DEFINITION ---
class GlacialLakeDataset(Dataset):
    def __init__(self, base_dir):
        self.images_dir = os.path.join(base_dir, "images")
        self.labels_dir = os.path.join(base_dir, "labels")
        self.filenames = [f for f in os.listdir(self.images_dir) if f.endswith(('.png', '.jpg'))]
        self.filenames.sort()

    def __len__(self): return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        img_path = os.path.join(self.images_dir, filename)
        lbl_path = os.path.join(self.labels_dir, filename)

        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        image = torch.tensor(image.astype(np.float32) / 255.0).permute(2, 0, 1)

        mask = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)
        mask = torch.tensor((mask > 127).astype(np.float32)).unsqueeze(0)

        return image, mask

# --- 3. ARCHITECTURE DEFINITION ---
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )
    def forward(self, x, skip=None):
        x = self.upsample(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = nn.functional.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class TransUNetHybrid(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=None)
        self.encoder1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.encoder2 = nn.Sequential(resnet.maxpool, resnet.layer1)
        self.encoder3 = resnet.layer2
        self.encoder4 = resnet.layer3
        self.encoder5 = resnet.layer4
        self.transformer_bridge = nn.Sequential(nn.Conv2d(2048, 512, kernel_size=1), nn.GroupNorm(32, 512), nn.ReLU(inplace=True))
        self.dec1, self.dec2 = DecoderBlock(512, 1024, 256), DecoderBlock(256, 512, 128)
        self.dec3, self.dec4 = DecoderBlock(128, 256, 64), DecoderBlock(64, 0, 32)
        self.final_head = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        c1, c2, c3, c4, c5 = self.encoder1(x), self.encoder2(self.encoder1(x)), self.encoder3(self.encoder2(self.encoder1(x))), self.encoder4(self.encoder3(self.encoder2(self.encoder1(x)))), self.encoder5(self.encoder4(self.encoder3(self.encoder2(self.encoder1(x)))))
        x = self.dec4(self.dec3(self.dec2(self.dec1(self.transformer_bridge(c5), c4), c3), c2))
        return nn.functional.interpolate(self.final_head(x), size=(512, 512), mode='bilinear', align_corners=True)

# --- 4. LOSS FUNCTIONS ---
class HybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, pred, target):
        p = torch.sigmoid(pred)
        dice = 1.0 - (2. * (p * target).sum(dim=(2, 3)) + 1e-5) / (p.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + 1e-5)
        return self.bce(pred, target) + dice.mean()

# --- 5. INITIALIZE AND LOAD EXISTING KNOWLEDGE ---
print("Preparing Datasets...")
dataset = GlacialLakeDataset(MASTER_DATASET_DIR)
train_set, val_set = random_split(dataset, [int(0.8 * len(dataset)), len(dataset) - int(0.8 * len(dataset))], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

print("Loading Brain...")
model = TransUNetHybrid().to(DEVICE)

# Fetch the perfectly formatted zip from Drive, copy locally, and rename to .pt
drive_weights_path = os.path.join(BASE_DRIVE_DIR, "best.pt.zip")
local_weights_path = "/content/structured_weights.pt"

if os.path.exists(drive_weights_path):
    print("Fetching structured weights from Drive...")
    shutil.copy(drive_weights_path, local_weights_path)

    try:
        model.load_state_dict(torch.load(local_weights_path, map_location=DEVICE))
        print("🚀 Previous knowledge successfully injected!")
    except Exception as e:
        print(f"❌ Failed to load weights. Error: {e}")
        print("Please ensure your 'archive' folder is at the absolute root of the zip file.")
else:
    print(f"WARNING: Could not find {drive_weights_path}. Training will start from scratch!")

criterion = HybridLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# --- 6. FINE-TUNING EXECUTION ---
best_loss = float('inf')
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_train_loss = 0.0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch} Train", leave=False):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), masks)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()

    scheduler.step()

    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch} Val", leave=False):
            running_val_loss += criterion(model(images.to(DEVICE)), masks.to(DEVICE)).item()

    val_loss = running_val_loss / len(val_loader)
    print(f"Epoch {epoch}/{EPOCHS} -> Train Loss: {running_train_loss / len(train_loader):.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_loss:
        print(f"✨ Validation improved! Saving new weights to {NEW_WEIGHTS_DIR}...")
        best_loss = val_loss
        torch.save(model.state_dict(), os.path.join(NEW_WEIGHTS_DIR, "best_finetuned.pt"))

Preparing Datasets...
Loading Brain...
Fetching structured weights from Drive...
🚀 Previous knowledge successfully injected!


Epoch 1 Train:   0%|          | 0/961 [00:00<?, ?it/s]

Epoch 1 Val:   0%|          | 0/241 [00:00<?, ?it/s]

Epoch 1/5 -> Train Loss: 0.5924 | Val Loss: 0.5662
✨ Validation improved! Saving new weights to /content/drive/MyDrive/GLOFEAGLES CHALLENGE/runs/train/fine_tuned/weights...


Epoch 2 Train:   0%|          | 0/961 [00:00<?, ?it/s]

...

Epoch 4 Val:   0%|          | 0/241 [00:00<?, ?it/s]

Epoch 4/5 -> Train Loss: 0.4697 | Val Loss: 0.5439


Epoch 5 Train:   0%|          | 0/961 [00:00<?, ?it/s]

Epoch 5 Val:   0%|          | 0/241 [00:00<?, ?it/s]

Epoch 5/5 -> Train Loss: 0.4564 | Val Loss: 0.4716
✨ Validation improved! Saving new weights to /content/drive/MyDrive/GLOFEAGLES CHALLENGE/runs/train/fine_tuned/weights...


ROUND 4: extracting >92% confidence images

In [ ]:
import os
import cv2
import cv2
import torch
import numpy as np
import shutil
from tqdm.notebook import tqdm

# --- 1. ROUND 4 PATHS ---
BASE_DRIVE_DIR = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE"
WEIGHTS_PATH = os.path.join(BASE_DRIVE_DIR, "runs/train/fine_tuned/weights/best_finetuned.pt")
UNLABELED_DIR = os.path.join(BASE_DRIVE_DIR, "Glofeagles Dataset unnanotated")
# Creating a new folder for Round 4
PSEUDO_DIR = os.path.join(BASE_DRIVE_DIR, "pseudo_labeled_round_4")

os.makedirs(os.path.join(PSEUDO_DIR, "images"), exist_ok=True)
os.makedirs(os.path.join(PSEUDO_DIR, "labels"), exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. LOAD LATEST FINE-TUNED MODEL ---
print("Loading fine-tuned model for Round 4 inference...")
model = TransUNetHybrid().to(DEVICE)
if os.path.exists(WEIGHTS_PATH):
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
    model.eval()
    print("✅ Model loaded successfully.")
else:
    raise FileNotFoundError(f"Could not find fine-tuned weights at {WEIGHTS_PATH}")

# --- 3. INFERENCE ---
unlabeled_files = [f for f in os.listdir(UNLABELED_DIR) if f.endswith(('.png', '.jpg'))]
image_scores = []

print(f"Scoring the remaining {len(unlabeled_files)} unlabeled images...")
with torch.no_grad():
    for filename in tqdm(unlabeled_files):
        img_path = os.path.join(UNLABELED_DIR, filename)
        img = cv2.imread(img_path)
        if img is None: continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor_img = torch.tensor(img_rgb.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

        probs = torch.sigmoid(model(tensor_img)).squeeze()
        binary_mask = (probs > 0.5).float()
        lake_pixels = probs[binary_mask == 1.0]

        confidence = lake_pixels.mean().item() * 100 if len(lake_pixels) > 50 else 0.0

        image_scores.append({
            "filename": filename,
            "confidence": confidence,
            "mask": binary_mask.cpu().numpy()
        })

# --- 4. STRICT 90% FILTER (ALL QUALIFYING IMAGES) ---
image_scores.sort(key=lambda x: x["confidence"], reverse=True)
confident_images = [x for x in image_scores if x["confidence"] >= 92.0]

print(f"\nFound {len(confident_images)} images with >= 92% confidence. Extracting ALL of them...")
for data in tqdm(confident_images, desc="Extracting Round 4 files"):
    filename = data["filename"]

    cv2.imwrite(os.path.join(PSEUDO_DIR, "labels", filename), (data["mask"] * 255).astype(np.uint8))
    shutil.copy(os.path.join(UNLABELED_DIR, filename), os.path.join(PSEUDO_DIR, "images", filename))
    os.remove(os.path.join(UNLABELED_DIR, filename))

print(f"Success! Round 4 generated {len(confident_images)} new high-quality samples.")
print(f"Remaining unlabeled pool: {len(os.listdir(UNLABELED_DIR))}")

Loading fine-tuned model for Round 4 inference...
✅ Model loaded successfully.
Scoring the remaining 139 unlabeled images...


  0%|          | 0/139 [00:00<?, ?it/s]


Found 22 images with >= 92% confidence. Extracting ALL of them...


Extracting Round 4 files:   0%|          | 0/22 [00:00<?, ?it/s]

Success! Round 4 generated 22 new high-quality samples.
Remaining unlabeled pool: 117


In [ ]:
import os
import cv2
import torch
import numpy as np
from tqdm.notebook import tqdm

# --- 1. PATHS ---
BASE_DRIVE_DIR = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE"
WEIGHTS_PATH = os.path.join(BASE_DRIVE_DIR, "runs/train/fine_tuned/weights/best_finetuned.pt")
UNLABELED_DIR = os.path.join(BASE_DRIVE_DIR, "Glofeagles Dataset unnanotated")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. LOAD LATEST FINE-TUNED MODEL ---
print("Loading fine-tuned model for residue analysis...")
model = TransUNetHybrid().to(DEVICE)
if os.path.exists(WEIGHTS_PATH):
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
    model.eval()
    print("✅ Model loaded successfully.")
else:
    raise FileNotFoundError(f"Could not find fine-tuned weights at {WEIGHTS_PATH}")

# --- 3. INFERENCE & SCORING ONLY (NO FILE MOVEMENT) ---
unlabeled_files = [f for f in os.listdir(UNLABELED_DIR) if f.endswith(('.png', '.jpg'))]
image_scores = []

print(f"Analyzing confidence for the remaining {len(unlabeled_files)} images...")
with torch.no_grad():
    for filename in tqdm(unlabeled_files, desc="Scanning Residue"):
        img_path = os.path.join(UNLABELED_DIR, filename)
        img = cv2.imread(img_path)
        if img is None: continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        tensor_img = torch.tensor(img_rgb.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

        probs = torch.sigmoid(model(tensor_img)).squeeze()
        binary_mask = (probs > 0.5).float()
        lake_pixels = probs[binary_mask == 1.0]

        confidence = lake_pixels.mean().item() * 100 if len(lake_pixels) > 50 else 0.0

        image_scores.append({
            "filename": filename,
            "confidence": confidence
        })

# --- 4. PRINT RAW DISTRIBUTION DATA ---
# Sort from highest confidence to lowest
image_scores.sort(key=lambda x: x["confidence"], reverse=True)

print("\n--- RESIDUE CONFIDENCE REPORT ---")
# Formatted strictly without spaces for clean data extraction
for item in image_scores:
    print(f"{item['filename']},{item['confidence']:.2f}")

print(f"\nTotal images analyzed: {len(image_scores)}")

Loading fine-tuned model for residue analysis...
✅ Model loaded successfully.
Analyzing confidence for the remaining 36 images...


Scanning Residue:   0%|          | 0/36 [00:00<?, ?it/s]


--- RESIDUE CONFIDENCE REPORT ---
292.png,62.84
365.png,60.97
554.png,60.82
163.png,60.51
79.png,0.00
9.png,0.00
154.png,0.00
341.png,0.00
557.png,0.00
275.png,0.00
267.png,0.00
276.png,0.00
221.png,0.00
310.png,0.00
384.png,0.00
295.png,0.00
45.png,0.00
96.png,0.00
293.png,0.00
349.png,0.00
500.png,0.00
151.png,0.00
543.png,0.00
199.png,0.00
27.png,0.00
457.png,0.00
59.png,0.00
561.png,0.00
393.png,0.00
235.png,0.00
32.png,0.00
283.png,0.00
156.png,0.00
7.png,0.00
110.png,0.00
485.png,0.00

Total images analyzed: 36


ROUND 4: 4807 + 103 + 103*10 =5940 images and Training

In [ ]:
import os
import cv2
import numpy as np
import albumentations as A
import zipfile
from tqdm.notebook import tqdm

# --- 1. EXACT PATHS ---
BASE_DRIVE_DIR = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE"
WEIGHTS_ZIP = os.path.join(BASE_DRIVE_DIR, "best.pt.zip")
PSEUDO_DIR = os.path.join(BASE_DRIVE_DIR, "pseudo_labeled_round_4")
MASTER_DATASET_DIR = os.path.join(BASE_DRIVE_DIR, "augmented_gt")
LOCAL_WEIGHTS_PATH = "/content/best.pt" # We extract to Colab's local drive for safe loading

# --- 2. EXTRACT WEIGHTS ---
print("Extracting best.pt.zip...")
if os.path.exists(WEIGHTS_ZIP):
    with zipfile.ZipFile(WEIGHTS_ZIP, 'r') as zip_ref:
        zip_ref.extractall("/content/")
    print("Weights extracted safely.")
else:
    print(f"Error: Could not find {WEIGHTS_ZIP}")

# --- 3. AUGMENTATION PIPELINE (Updated for Albumentations v1.4+) ---
transform = A.Compose([
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.8),
    A.HueSaturationValue(hue_shift_limit=30, sat_shift_limit=40, val_shift_limit=20, p=0.8),
    A.OneOf([
        A.Affine(rotate=0, p=1.0), A.Affine(rotate=45, p=1.0), A.Affine(rotate=90, p=1.0),
        A.Affine(rotate=135, p=1.0), A.Affine(rotate=180, p=1.0), A.Affine(rotate=225, p=1.0),
        A.Affine(rotate=270, p=1.0), A.Affine(rotate=315, p=1.0),
    ], p=0.8),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),

    # Modernized parameters
    A.RandomFog(p=0.3),
    A.GaussNoise(p=0.3),
    A.ElasticTransform(alpha=1, sigma=50, p=0.3), # Removed deprecated alpha_affine
    A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 32), hole_width_range=(8, 32), fill=0, mask_fill=0, p=0.1),
    A.GridDistortion(num_steps=5, distort_limit=0.2, p=0.1),
    A.RandomResizedCrop(size=(512, 512), scale=(0.85, 1.0), ratio=(0.9, 1.1), p=0.1) # Combined height/width into size tuple
])

# --- 4. MERGE DIRECTLY INTO MASTER DATASET ---
pseudo_images_dir = os.path.join(PSEUDO_DIR, "images")
pseudo_labels_dir = os.path.join(PSEUDO_DIR, "labels")

master_images_dir = os.path.join(MASTER_DATASET_DIR, "images")
master_labels_dir = os.path.join(MASTER_DATASET_DIR, "labels")

pseudo_files = [f for f in os.listdir(pseudo_images_dir) if f.endswith(('.png', '.jpg'))]

print(f"Augmenting {len(pseudo_files)} pseudo-labeled images into the master dataset...")
for filename in tqdm(pseudo_files):
    img_path = os.path.join(pseudo_images_dir, filename)
    lbl_path = os.path.join(pseudo_labels_dir, filename)

    image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)

    # Save original pseudo-label with a prefix to prevent name clashes
    base_name = os.path.splitext(filename)[0]
    cv2.imwrite(os.path.join(master_images_dir, f"pseudo_{base_name}_aug_0.png"), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(master_labels_dir, f"pseudo_{base_name}_aug_0.png"), mask)

    # Generate 10 variations
    for i in range(1, 11):
        augmented = transform(image=image, mask=mask)
        new_filename = f"pseudo_{base_name}_aug_{i}.png"

        cv2.imwrite(os.path.join(master_images_dir, new_filename), cv2.cvtColor(augmented['image'], cv2.COLOR_RGB2BGR))
        cv2.imwrite(os.path.join(master_labels_dir, new_filename), augmented['mask'])

print(f"Success! Master dataset now contains {len(os.listdir(master_images_dir))} total images.")

Extracting best.pt.zip...
Weights extracted safely.
Augmenting 103 pseudo-labeled images into the master dataset...


/tmp/ipykernel_3517/1269355254.py:40: UserWarning: Argument(s) 'mask_fill' are not valid for transform CoarseDropout
  A.CoarseDropout(num_holes_range=(1, 4), hole_height_range=(8, 32), hole_width_range=(8, 32), fill=0, mask_fill=0, p=0.1),


  0%|          | 0/103 [00:00<?, ?it/s]

Success! Master dataset now contains 5940 total images.


In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.models as models
from tqdm.notebook import tqdm
import shutil

# --- 1. FINE-TUNING HYPERPARAMETERS & PATHS ---
BASE_DRIVE_DIR = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE"
MASTER_DATASET_DIR = os.path.join(BASE_DRIVE_DIR, "augmented_gt")
NEW_WEIGHTS_DIR = os.path.join(BASE_DRIVE_DIR, "runs/train/fine_tuned/weights")

os.makedirs(NEW_WEIGHTS_DIR, exist_ok=True)

EPOCHS = 1                  # Fine-tuning requires fewer epochs
BATCH_SIZE = 5
LEARNING_RATE = 1.2e-5         # Low learning rate to prevent Catastrophic Forgetting
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. DATASET DEFINITION ---
class GlacialLakeDataset(Dataset):
    def __init__(self, base_dir):
        self.images_dir = os.path.join(base_dir, "images")
        self.labels_dir = os.path.join(base_dir, "labels")
        self.filenames = [f for f in os.listdir(self.images_dir) if f.endswith(('.png', '.jpg'))]
        self.filenames.sort()

    def __len__(self): return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        img_path = os.path.join(self.images_dir, filename)
        lbl_path = os.path.join(self.labels_dir, filename)

        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        image = torch.tensor(image.astype(np.float32) / 255.0).permute(2, 0, 1)

        mask = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)
        mask = torch.tensor((mask > 127).astype(np.float32)).unsqueeze(0)

        return image, mask

# --- 3. ARCHITECTURE DEFINITION ---
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )
    def forward(self, x, skip=None):
        x = self.upsample(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = nn.functional.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class TransUNetHybrid(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=None)
        self.encoder1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.encoder2 = nn.Sequential(resnet.maxpool, resnet.layer1)
        self.encoder3 = resnet.layer2
        self.encoder4 = resnet.layer3
        self.encoder5 = resnet.layer4
        self.transformer_bridge = nn.Sequential(nn.Conv2d(2048, 512, kernel_size=1), nn.GroupNorm(32, 512), nn.ReLU(inplace=True))
        self.dec1, self.dec2 = DecoderBlock(512, 1024, 256), DecoderBlock(256, 512, 128)
        self.dec3, self.dec4 = DecoderBlock(128, 256, 64), DecoderBlock(64, 0, 32)
        self.final_head = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        c1, c2, c3, c4, c5 = self.encoder1(x), self.encoder2(self.encoder1(x)), self.encoder3(self.encoder2(self.encoder1(x))), self.encoder4(self.encoder3(self.encoder2(self.encoder1(x)))), self.encoder5(self.encoder4(self.encoder3(self.encoder2(self.encoder1(x)))))
        x = self.dec4(self.dec3(self.dec2(self.dec1(self.transformer_bridge(c5), c4), c3), c2))
        return nn.functional.interpolate(self.final_head(x), size=(512, 512), mode='bilinear', align_corners=True)

# --- 4. LOSS FUNCTIONS ---
class HybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, pred, target):
        p = torch.sigmoid(pred)
        dice = 1.0 - (2. * (p * target).sum(dim=(2, 3)) + 1e-5) / (p.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + 1e-5)
        return self.bce(pred, target) + dice.mean()

# --- 5. INITIALIZE AND LOAD EXISTING KNOWLEDGE ---
print("Preparing Datasets...")
dataset = GlacialLakeDataset(MASTER_DATASET_DIR)
train_set, val_set = random_split(dataset, [int(0.8 * len(dataset)), len(dataset) - int(0.8 * len(dataset))], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

print("Loading Brain...")
model = TransUNetHybrid().to(DEVICE)

# Fetch the perfectly formatted zip from Drive, copy locally, and rename to .pt
drive_weights_path = os.path.join(BASE_DRIVE_DIR, "best.pt.zip")
local_weights_path = "/content/structured_weights.pt"

if os.path.exists(drive_weights_path):
    print("Fetching structured weights from Drive...")
    shutil.copy(drive_weights_path, local_weights_path)

    try:
        model.load_state_dict(torch.load(local_weights_path, map_location=DEVICE))
        print("🚀 Previous knowledge successfully injected!")
    except Exception as e:
        print(f" Failed to load weights. Error: {e}")
        print("Please ensure your 'archive' folder is at the absolute root of the zip file.")
else:
    print(f"WARNING: Could not find {drive_weights_path}. Training will start from scratch!")

criterion = HybridLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# --- 6. FINE-TUNING EXECUTION ---
best_loss = float('inf')
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_train_loss = 0.0
    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch} Train", leave=False):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), masks)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()

    scheduler.step()

    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Epoch {epoch} Val", leave=False):
            running_val_loss += criterion(model(images.to(DEVICE)), masks.to(DEVICE)).item()

    val_loss = running_val_loss / len(val_loader)
    print(f"Epoch {epoch}/{EPOCHS} -> Train Loss: {running_train_loss / len(train_loader):.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_loss:
        print(f"✨ Validation improved! Saving new weights to {NEW_WEIGHTS_DIR}...")
        best_loss = val_loss
        torch.save(model.state_dict(), os.path.join(NEW_WEIGHTS_DIR, "best_finetuned.pt"))

Preparing Datasets...
Loading Brain...
Fetching structured weights from Drive...
🚀 Previous knowledge successfully injected!


Epoch 1 Train:   0%|          | 0/950 [00:00<?, ?it/s]

Epoch 1 Val:   0%|          | 0/238 [00:00<?, ?it/s]

Epoch 1/1 -> Train Loss: 0.6391 | Val Loss: 0.6027
✨ Validation improved! Saving new weights to /content/drive/MyDrive/GLOFEAGLES CHALLENGE/runs/train/fine_tuned/weights...


In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision.models as models
from tqdm.notebook import tqdm

# --- 1. CONFIGURATION FOR 2 MORE EPOCHS ---
BASE_DRIVE_DIR = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE"
MASTER_DATASET_DIR = os.path.join(BASE_DRIVE_DIR, "augmented_gt")
NEW_WEIGHTS_DIR = os.path.join(BASE_DRIVE_DIR, "runs/train/fine_tuned/weights")

EPOCHS = 1                   # Strictly 2 more epochs to check for data plateau
BATCH_SIZE = 5               # Keeping your optimized batch size
LEARNING_RATE = 1.2e-5         # Safe, micro-adjustment learning rate
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 2. DATASET DEFINITION ---
class GlacialLakeDataset(Dataset):
    def __init__(self, base_dir):
        self.images_dir = os.path.join(base_dir, "images")
        self.labels_dir = os.path.join(base_dir, "labels")
        self.filenames = [f for f in os.listdir(self.images_dir) if f.endswith(('.png', '.jpg'))]
        self.filenames.sort()

    def __len__(self): return len(self.filenames)

    def __getitem__(self, idx):
        filename = self.filenames[idx]
        img_path = os.path.join(self.images_dir, filename)
        lbl_path = os.path.join(self.labels_dir, filename)

        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        image = torch.tensor(image.astype(np.float32) / 255.0).permute(2, 0, 1)

        mask = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)
        mask = torch.tensor((mask > 127).astype(np.float32)).unsqueeze(0)

        return image, mask

# --- 3. ARCHITECTURE DEFINITION ---
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )
    def forward(self, x, skip=None):
        x = self.upsample(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = nn.functional.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class TransUNetHybrid(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=None)
        self.encoder1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.encoder2 = nn.Sequential(resnet.maxpool, resnet.layer1)
        self.encoder3 = resnet.layer2
        self.encoder4 = resnet.layer3
        self.encoder5 = resnet.layer4
        self.transformer_bridge = nn.Sequential(nn.Conv2d(2048, 512, kernel_size=1), nn.GroupNorm(32, 512), nn.ReLU(inplace=True))
        self.dec1, self.dec2 = DecoderBlock(512, 1024, 256), DecoderBlock(256, 512, 128)
        self.dec3, self.dec4 = DecoderBlock(128, 256, 64), DecoderBlock(64, 0, 32)
        self.final_head = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        c1, c2, c3, c4, c5 = self.encoder1(x), self.encoder2(self.encoder1(x)), self.encoder3(self.encoder2(self.encoder1(x))), self.encoder4(self.encoder3(self.encoder2(self.encoder1(x)))), self.encoder5(self.encoder4(self.encoder3(self.encoder2(self.encoder1(x)))))
        x = self.dec4(self.dec3(self.dec2(self.dec1(self.transformer_bridge(c5), c4), c3), c2))
        return nn.functional.interpolate(self.final_head(x), size=(512, 512), mode='bilinear', align_corners=True)

# --- 4. LOSS FUNCTIONS ---
class HybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
    def forward(self, pred, target):
        p = torch.sigmoid(pred)
        dice = 1.0 - (2. * (p * target).sum(dim=(2, 3)) + 1e-5) / (p.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + 1e-5)
        return self.bce(pred, target) + dice.mean()

# --- 5. INITIALIZE AND RESUME FROM LATEST WEIGHTS ---
print("Preparing Datasets...")
dataset = GlacialLakeDataset(MASTER_DATASET_DIR)
train_set, val_set = random_split(dataset, [int(0.8 * len(dataset)), len(dataset) - int(0.8 * len(dataset))], generator=torch.Generator().manual_seed(42))
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

print("Loading Brain...")
model = TransUNetHybrid().to(DEVICE)

# CRITICAL: Directly load the latest .pt file instead of the base zip
latest_weights_path = os.path.join(NEW_WEIGHTS_DIR, "best_finetuned.pt")

if os.path.exists(latest_weights_path):
    print(f"🔄 Loading existing fine-tuned weights from: {latest_weights_path}")
    model.load_state_dict(torch.load(latest_weights_path, map_location=DEVICE))
    print("🚀 Resuming seamlessly from your previous checkpoint!")
else:
    raise FileNotFoundError(f"Could not find fine-tuned weights at {latest_weights_path}. Verify the path before running.")

criterion = HybridLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# --- 6. EXECUTE EXTRA EPOCHS ---
best_loss = float('inf')
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_train_loss = 0.0
    for images, masks in tqdm(train_loader, desc=f"Extension Epoch {epoch} Train", leave=False):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), masks)
        loss.backward()
        optimizer.step()
        running_train_loss += loss.item()

    scheduler.step()

    model.eval()
    running_val_loss = 0.0
    with torch.no_grad():
        for images, masks in tqdm(val_loader, desc=f"Extension Epoch {epoch} Val", leave=False):
            running_val_loss += criterion(model(images.to(DEVICE)), masks.to(DEVICE)).item()

    val_loss = running_val_loss / len(val_loader)
    print(f"Epoch {epoch}/{EPOCHS} -> Train Loss: {running_train_loss / len(train_loader):.4f} | Val Loss: {val_loss:.4f}")

    if val_loss < best_loss:
        print(f"✨ Validation improved during extension! Saving updated weights...")
        best_loss = val_loss
        torch.save(model.state_dict(), os.path.join(NEW_WEIGHTS_DIR, "best_finetuned.pt"))

Preparing Datasets...
Loading Brain...
🔄 Loading existing fine-tuned weights from: /content/drive/MyDrive/GLOFEAGLES CHALLENGE/runs/train/fine_tuned/weights/best_finetuned.pt
🚀 Resuming seamlessly from your previous checkpoint!


Extension Epoch 1 Train:   0%|          | 0/950 [00:00<?, ?it/s]

Extension Epoch 1 Val:   0%|          | 0/238 [00:00<?, ?it/s]

Epoch 1/1 -> Train Loss: 0.5731 | Val Loss: 0.5861
✨ Validation improved during extension! Saving updated weights...


In [ ]:
import os
import time
import cv2
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, jaccard_score, precision_score, recall_score, f1_score, cohen_kappa_score
import matplotlib.pyplot as plt

# --- 1. REPRODUCIBILITY SETUP (ENVIRONMENT DETERMINISM) ---
def enforce_reproducibility(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

enforce_reproducibility(42)

print("=== REPRODUCIBILITY MATRIX ===")
print(f"• PyTorch Version : {torch.__version__}")
print(f"• CUDA Available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"• GPU Device       : {torch.cuda.get_device_name(0)}")
print("=============================\n")

# --- 2. CONFIGURATION PATHS ---
BASE_DRIVE_DIR = "/content/drive/MyDrive/GLOFEAGLES CHALLENGE"
WEIGHTS_PATH = os.path.join(BASE_DRIVE_DIR, "runs/train/fine_tuned/weights/best_finetuned.pt")
VAL_DIR = os.path.join(BASE_DRIVE_DIR, "labeled dataset")

images_dir = os.path.join(VAL_DIR, "images")
labels_dir = os.path.join(VAL_DIR, "labels")
VIS_OUTPUT_DIR = os.path.join(BASE_DRIVE_DIR, "report_visualizations")
os.makedirs(VIS_OUTPUT_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- 3. ARCHITECTURE DEFINITION ---
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True)
        )
    def forward(self, x, skip=None):
        x = self.upsample(x)
        if skip is not None:
            if x.shape[2:] != skip.shape[2:]:
                x = nn.functional.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
            x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class TransUNetHybrid(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet50(weights=None)
        self.encoder1 = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.encoder2 = nn.Sequential(resnet.maxpool, resnet.layer1)
        self.encoder3 = resnet.layer2
        self.encoder4 = resnet.layer3
        self.encoder5 = resnet.layer4
        self.transformer_bridge = nn.Sequential(nn.Conv2d(2048, 512, kernel_size=1), nn.GroupNorm(32, 512), nn.ReLU(inplace=True))
        self.dec1, self.dec2 = DecoderBlock(512, 1024, 256), DecoderBlock(256, 512, 128)
        self.dec3, self.dec4 = DecoderBlock(128, 256, 64), DecoderBlock(64, 0, 32)
        self.final_head = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, x):
        c1, c2, c3, c4, c5 = self.encoder1(x), self.encoder2(self.encoder1(x)), self.encoder3(self.encoder2(self.encoder1(x))), self.encoder4(self.encoder3(self.encoder2(self.encoder1(x)))), self.encoder5(self.encoder4(self.encoder3(self.encoder2(self.encoder1(x)))))
        x = self.dec4(self.dec3(self.dec2(self.dec1(self.transformer_bridge(c5), c4), c3), c2))
        return nn.functional.interpolate(self.final_head(x), size=(512, 512), mode='bilinear', align_corners=True)

# --- 4. MODEL INITIALIZATION ---
model = TransUNetHybrid().to(DEVICE)
if os.path.exists(WEIGHTS_PATH):
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
    model.eval()
    print("✅ Model loaded successfully for advanced benchmarking.")
else:
    raise FileNotFoundError(f"Missing fine-tuned weights file at {WEIGHTS_PATH}")

# --- 5. METRIC STORAGE & BENCHMARKING LOOP ---
val_files = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg'))]

precisions, recalls, f1_scores, kappas, accuracies, jaccards = [], [], [], [], [], []
per_image_robustness_log = []

start_time = time.time()

print(f"Evaluating {len(val_files)} verification scenes and rendering visualization reports...")
with torch.no_grad():
    for filename in tqdm(val_files, desc="Running Analysis"):
        img_path = os.path.join(images_dir, filename)
        lbl_path = os.path.join(labels_dir, filename)

        img_bgr = cv2.imread(img_path)
        gt_mask = cv2.imread(lbl_path, cv2.IMREAD_GRAYSCALE)
        if img_bgr is None or gt_mask is None: continue

        y_true = (gt_mask > 127).astype(np.uint8).flatten()

        # Inference pipeline
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        tensor_img = torch.tensor(img_rgb.astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(DEVICE)

        probs = torch.sigmoid(model(tensor_img)).squeeze().cpu().numpy()
        pred_mask_2d = (probs > 0.5).astype(np.uint8)
        y_pred = pred_mask_2d.flatten()

        # Metric Calculations
        acc = accuracy_score(y_true, y_pred)
        jaccard = jaccard_score(y_true, y_pred, zero_division=1)
        p = precision_score(y_true, y_pred, zero_division=1)
        r = recall_score(y_true, y_pred, zero_division=1)
        f1 = f1_score(y_true, y_pred, zero_division=1)
        kappa = cohen_kappa_score(y_true, y_pred)

        if np.isnan(kappa):
            kappa = 1.0 if np.array_equal(y_true, y_pred) else 0.0

        # Append to global collectors
        accuracies.append(acc)
        jaccards.append(jaccard)
        precisions.append(p)
        recalls.append(r)
        f1_scores.append(f1)
        kappas.append(kappa)

        # Log for Robustness Analysis
        per_image_robustness_log.append({
            "filename": filename,
            "jaccard": jaccard,
            "f1": f1
        })

        # --- 6. AUTOMATED VISUALIZATION GENERATION ---
        # Construct side-by-side visual evaluation assets
        overlay = img_rgb.copy()
        # Highlight correct predictions (True Positives) in Bright Cyan
        overlay[(pred_mask_2d == 1) & (gt_mask > 127)] = overlay[(pred_mask_2d == 1) & (gt_mask > 127)] * 0.4 + np.array([0, 255, 255]) * 0.6
        # Highlight False Positives (Hallucinations/Shadow errors) in Red
        overlay[(pred_mask_2d == 1) & (gt_mask <= 127)] = overlay[(pred_mask_2d == 1) & (gt_mask <= 127)] * 0.3 + np.array([255, 0, 0]) * 0.7

        fig, axs = plt.subplots(1, 3, figsize=(15, 5))
        axs[0].imshow(img_rgb)
        axs[0].set_title("Input Imagery")
        axs[0].axis('off')

        axs[1].imshow(gt_mask, cmap='gray')
        axs[1].set_title("Ground Truth Target")
        axs[1].axis('off')

        axs[2].imshow(overlay)
        axs[2].set_title(f"Model Prediction Overlay\nIoU/Jaccard: {jaccard*100:.1f}%")
        axs[2].axis('off')

        plt.tight_layout()
        plt.savefig(os.path.join(VIS_OUTPUT_DIR, f"eval_{filename}"), bbox_inches='tight', dpi=150)
        plt.close()

execution_time = time.time() - start_time

# --- 7. REPRODUCIBLE REPORT MATRIX CARD ---
print("\n=============================================")
print("       EXPANDED PERFORMANCE REPORT CARD      ")
print("=============================================")
print(f"• Overall Pixel Accuracy : {np.mean(accuracies) * 100:.2f}%")
print(f"• Jaccard Index (mIoU)   : {np.mean(jaccards) * 100:.2f}%")
print(f"• Mean F1 Score (Dice)   : {np.mean(f1_scores) * 100:.2f}%")
print(f"• Mean Precision         : {np.mean(precisions) * 100:.2f}%")
print(f"• Mean Recall            : {np.mean(recalls) * 100:.2f}%")
print(f"• Cohen's Kappa          : {np.mean(kappas):.4f}")
print("---------------------------------------------")
print(f"• Total Benchmarking Time: {execution_time:.2f} seconds")
print(f"• Visual Proofs Exported : {VIS_OUTPUT_DIR}")
print("=============================================\n")

# --- 8. ROBUSTNESS & CORNER-CASE ANALYSIS ---
# Sort images by lowest Jaccard index to surface challenging scenes
per_image_robustness_log.sort(key=lambda x: x["jaccard"])

print("=============================================")
print("      CRITICAL ROBUSTNESS ANALYSIS (WORST 5) ")
print("=============================================")
print("Review these specific scenes to document environmental failure modes:")
for i in range(min(5, len(per_image_robustness_log))):
    case = per_image_robustness_log[i]
    print(f" {i+1}. Scene ID: {case['filename']} | Jaccard IoU: {case['jaccard']*100:.2f}%")
print("=============================================")

=== REPRODUCIBILITY MATRIX ===
• PyTorch Version : 2.11.0+cpu
• CUDA Available   : False

✅ Model loaded successfully for advanced benchmarking.
Evaluating 60 verification scenes and rendering visualization reports...


Running Analysis:   0%|          | 0/60 [00:00<?, ?it/s]


       EXPANDED PERFORMANCE REPORT CARD      
• Overall Pixel Accuracy : 99.69%
• Jaccard Index (mIoU)   : 77.20%
• Mean F1 Score (Dice)   : 83.16%
• Mean Precision         : 84.01%
• Mean Recall            : 84.08%
• Cohen's Kappa          : 0.8302
---------------------------------------------
• Total Benchmarking Time: 282.79 seconds
• Visual Proofs Exported : /content/drive/MyDrive/GLOFEAGLES CHALLENGE/report_visualizations

      CRITICAL ROBUSTNESS ANALYSIS (WORST 5) 
Review these specific scenes to document environmental failure modes:
 1. Scene ID: 202.png | Jaccard IoU: 0.00%
 2. Scene ID: 423.png | Jaccard IoU: 0.00%
 3. Scene ID: 540.png | Jaccard IoU: 0.00%
 4. Scene ID: 544.png | Jaccard IoU: 0.00%
 5. Scene ID: 39.png | Jaccard IoU: 13.00%
